# Atividade: Predição de Diabetes — Baseline com Remoção de Registros Incompletos (Complete Case Analysis)

Experimento baseline focado no descarte de instâncias com dados faltantes (*Pima Indians Diabetes*):
1. **Estratégia:** Eliminação direta de todas as linhas que apresentam zeros em atributos fisiologicamente não nulos (Glicose, Pressão, etc.).
2. **Padronização:** `StandardScaler`.
3. **Classificador:** k-NN baseline (k=3).
4. **Objetivo Comparativo:** Avaliar a degradação amostral e a perda de representatividade ao descartar instâncias em vez de aplicar imputação.


### 1. Importações e Configurações
Configuramos as bibliotecas e a chave de acesso da equipe.

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import requests

# --- CONFIGURAÇÃO ---
DEV_KEY = "2_computeiros_e_um_matemagico"
URL = "https://aydanomachado.com/mlclass/01_Preprocessing.php"

feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

### 2. Carga dos Dados
Lemos os arquivos CSV originais.

In [ ]:
df_train = pd.read_csv('diabetes_dataset.csv')
df_app = pd.read_csv('diabetes_app.csv')

print('Dados de treino carregados:', df_train.shape)
print('Dados de aplicação carregados:', df_app.shape)

### 3. Pré-processamento de Dados
**Nesta seção, vamos implementar as técnicas de pré-processamento**

In [ ]:
df = df_train.copy()
df.head()

df.drop(columns=['Insulin', 'SkinThickness'])

# Mostra a quantidade de valores vazios por coluna
print(df.isnull().sum())

In [ ]:
# lista das colunas que tem NaN
colunas_com_vazios = ['Glucose', 'BloodPressure', 'BMI']

# preencher com base na média agrupado por OutCome
for coluna in colunas_com_vazios:
  media_coluna_agrupada = df.groupby('Outcome')[coluna].transform('median')
  df[coluna] = df[coluna].fillna(media_coluna_agrupada)

# como os dois tem MUITOS valores faltantes, melhor tirar
df.drop(columns=['Insulin', 'SkinThickness'])

# pra verificar se preencher mesmo
print(df.isnull().sum())

In [5]:
features_novas = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']
X = df[features_novas]
y = df['Outcome']

X_app = df_app[features_novas]

In [8]:
#padronizar com média
scaler = StandardScaler()

#treina a régua no X e já transforma o X
X_escalonado = scaler.fit_transform(X)

#apenas aplica a mesma coisa no X_app
X_app_escalonado = scaler.transform(X_app)

In [9]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X_escalonado, y, test_size=0.2, random_state=42)

knn_teste = KNeighborsClassifier(n_neighbors=3)
knn_teste.fit(X_treino, y_treino)
previsoes_teste = knn_teste.predict(X_teste)

acuracia_local = accuracy_score(y_teste, previsoes_teste) * 100
print(f" ACURÁCIA DA SUA ESTRATÉGIA (K=3): {acuracia_local:.2f}%")
print(classification_report(y_teste, previsoes_teste))

 ACURÁCIA DA SUA ESTRATÉGIA (K=3): 79.13%
              precision    recall  f1-score   support

           0       0.81      0.88      0.84        72
           1       0.76      0.65      0.70        43

    accuracy                           0.79       115
   macro avg       0.78      0.76      0.77       115
weighted avg       0.79      0.79      0.79       115



#### 3.5. Preparação Final das Variáveis
Ajustem as variáveis X e X_app após todas as transformações acima.

In [ ]:
X = df_train[feature_cols]
y = df_train['Outcome']

X_app = df_app[feature_cols]

print("Pré-processamento concluído.")

### 4. Treinamento e Predição
O modelo k-NN é treinado com os dados processados e as predições são geradas.

In [ ]:
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X, y)

y_pred = neigh.predict(X_app)
print("Predições geradas com sucesso.")

### 5. Envio de Resultados
Submissão automática para o servidor para checagem da acurácia.

In [ ]:
dados_envio = {
    'dev_key': DEV_KEY,
    'predictions': pd.Series(y_pred).to_json(orient='values')
}

print('Enviando para o servidor...')
r = requests.post(url = URL, data = dados_envio)
print("Resposta do servidor:\n", r.text)